In [15]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os


In [16]:
import cv2
import numpy as np

def custom_preprocessing(img):
    # img comes as numpy array (224,224,3)
    
    # Convert to grayscale
    img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    
    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img = clahe.apply(img.astype(np.uint8))
    
    # Gaussian blur
    img = cv2.GaussianBlur(img, (3,3), 0)
    
    # Normalize
    img = img / 255.0
    
    # Convert back to 3-channel (for ResNet/VGG)
    img = np.stack((img,)*3, axis=-1)
    
    return img


In [17]:
train_dir = "../dataset/train"
val_dir   = "../dataset/validation"
test_dir  = "../dataset/test"

train_gen = ImageDataGenerator(
    preprocessing_function=custom_preprocessing,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(
    preprocessing_function=custom_preprocessing
)

test_gen = ImageDataGenerator(
    preprocessing_function=custom_preprocessing
)


In [18]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Get class labels from training data
class_indices = train_data.class_indices
print(class_indices)  # {'Normal': 0, 'Stroke': 1}

labels = train_data.classes

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(class_weights))
print("Class weights:", class_weights)


{'Normal': 0, 'Stroke': 1}
Class weights: {0: 0.8477460901563938, 1: 1.218915343915344}


In [19]:
train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=(224,224),
    batch_size=16,
    class_mode='binary'
)

val_data = val_gen.flow_from_directory(
    val_dir,
    target_size=(224,224),
    batch_size=16,
    class_mode='binary'
)

test_data = test_gen.flow_from_directory(
    test_dir,
    target_size=(224,224),
    batch_size=16,
    class_mode='binary',
    shuffle=False
)


Found 1843 images belonging to 2 classes.
Found 235 images belonging to 2 classes.
Found 437 images belonging to 2 classes.


In [20]:
images, labels = next(train_data)

print(images.shape)     # (16, 224, 224, 3)
print(images.min(), images.max())  # ~0 to 1


(16, 224, 224, 3)
0.007843138 1.0
